#### Imports

In [1]:
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt
import struct
import math

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm
import time
import random

#### Useful functions

In [34]:
### Takes a box and splits it into cubes (blocks) of a specified dimension. Returns
### a list of numpy arrays, each representing one block of the box. If block_dim
### does not divide equally into the volume dimensions, creates blocks that extend past the
### dimensions of the volume, and populates the coordinates outside the volume with chosen 
### quantity junk, which is an input to the function
def to_blocks(box, block_dim, junk):
    
    xdim = box.shape[0]
    ydim = box.shape[1]
    zdim = box.shape[2]

    x_steps = int(xdim // block_dim)
    if xdim % block_dim != 0:
        x_steps += 1
    if xdim < block_dim:
        x_steps = 1
    y_steps = int(ydim / block_dim)
    if ydim % block_dim != 0:
        y_steps += 1
    if ydim < block_dim:
        y_steps = 1
    z_steps = int(zdim / block_dim)
    if zdim % block_dim != 0:
        z_steps += 1
    if zdim < block_dim:
        z_steps = 1

    blocks = []
    
    for k in range(z_steps):
        for j in range(y_steps):
            for i in range(x_steps):

                x_lo, y_lo, z_lo = int(i * block_dim), int(j * block_dim), int(k * block_dim)
                x_hi, y_hi, z_hi = min(x_lo + block_dim, xdim), min(y_lo + block_dim, ydim), min(z_lo + block_dim, zdim)

                # create a block, fill with junk
                block = np.full((block_dim, block_dim, block_dim), junk)

                # slice the block from the original box
                block[:x_hi - x_lo, :y_hi - y_lo, :z_hi - z_lo] = box[x_lo:x_hi, y_lo:y_hi, z_lo:z_hi]

                block = block.reshape(-1)
                blocks.append(block)

    return blocks



### Reads binary files in input_dir for the levels between min_level and max_level,
### and returns them in a list of numpy arrays, one for each chunk of data (box).
### Also returns a list of the tuples of the location and dimension of each
### box. Finally, returns a list of the number of boxes at each level.
def process_data(dir_prefix, timestep, min_level, max_level, component):
    
    # create output lists
    boxes, locations, dimensions, box_counts = [], [], [], []

    print("Processing data...")
    for l in tqdm(range(min_level, max_level+1)): # iterate over each level
        filename = f"{dir_prefix}-{l}/{timestep}-wholeNewFormat-{component}-{l}.raw"

        # Keeps track of how many boxes are at each level
        box_count = 0
        boxes_l, locations_l, dimensions_l = [], [], []
        
        with open(filename, "rb") as file:
            while True: # Iterate through all the data at the current level until none is left
                test = file.read(4)
                if len(test) < 4: # Break the loop if all data has been read
                    break
                # Read the location of the box
                x = int(struct.unpack('<f', test)[0])
                y = int(struct.unpack('<f', file.read(4))[0])
                z = int(struct.unpack('<f', file.read(4))[0])
                locations_l.append((x, y, z))

                # Read the dimensions of the box
                xdim = int(struct.unpack('<f', file.read(4))[0])
                ydim = int(struct.unpack('<f', file.read(4))[0])
                zdim = int(struct.unpack('<f', file.read(4))[0])
                dimensions_l.append((xdim, ydim, zdim))

                # Read the data in the box
                box = np.empty((xdim, ydim, zdim))
                for k in range(zdim):
                    for j in range(ydim):
                        for i in range(xdim):
                            box[i][j][k] = struct.unpack('<f', file.read(4))[0]
                boxes_l.append(box)
                box_count += 1

        boxes.append(boxes_l)
        locations.append(locations_l)
        dimensions.append(dimensions_l)
        box_counts.append(box_count)

    return boxes, locations, dimensions, box_counts



### Takes a set of read boxes, and breaks them into blocks of a specified
### dimension for use in creating a codebook.
def create_samples(boxes, block_dim, junk):

    print("Creating samples...")
    
    # Check if block_dim is a power of 2
    if not math.log2(block_dim).is_integer():
        print("Invalid block_dim! Dimension must be a power of 2.")
        return
    
    samples = []

    for l in tqdm(range(len(boxes))): # iterate through all levels
        boxes_l = boxes[l]
        samples_l = []
    
        for box in boxes_l: # Iterate through each box at the current level
            # Create blocks of the desired dimension from the box
            blocks = to_blocks(box, block_dim, junk)
            samples_l.append(blocks)

        samples.append(samples_l)

    return samples



### Randomly initializes num_codewords codewords by copying them at random
### from an input list of blocks
def initialize_codewords_random(blocks, num_codewords):
    random_indices = random.sample(range(len(blocks)), num_codewords)
    codewords = blocks[random_indices]
    return codewords


### Initializes num_codewords codewords according to the method laid out
### by Hu et al. (2015)
def initialize_codewords_enhanced(blocks, num_codewords, block_dim):

    # Sort by distance to origin
    origin = cp.zeros((block_dim**3), dtype=cp.float32)
    distances = cp.linalg.norm(blocks - origin, axis=1)

    # Sort by sum of values
    sums = cp.sum(blocks, axis=1)

    # Sort indices based on distances and sums
    sorted_d1 = cp.argsort(distances)
    sorted_d2 = cp.argsort(sums)

    # Find initial codewords
    # Split sorted indices into equal parts
    d1_subs = cp.array_split(sorted_d1, num_codewords)
    d2_subs = cp.array_split(sorted_d2, num_codewords)

    codeword_inds = []
    for i in range(num_codewords):
        d1_sub = d1_subs[i]
        d2_sub = d2_subs[i]

        intersection = cp.intersect1d(d1_sub, d2_sub)

        if intersection.size == 0:
            codeword_idx = cp.median(d1_sub)
        else:
            codeword_idx = cp.median(intersection)

        codeword_inds.append(int(codeword_idx))

    codewords = blocks[codeword_inds]

    return codewords



### Note, assumes input blocks and codewords to be CuPy arrays. Iterates through
### blocks, assigning each one to the nearest codeword by Euclidian distance.
def assign_to_codeword(blocks, codewords, batch_size=1000):

    num_blocks = blocks.shape[0]
    num_codewords = codewords.shape[0]
    assignments = cp.zeros(num_blocks, dtype=cp.int32)

    for i in range(0, num_blocks, batch_size): # Iterate through the necessary number of
                                               # batches of blocks
        # Take a batch of blocks
        blocks_batch = blocks[i:i+batch_size]

        # Reshape blocks and codewords for use in linalg.norm
        blocks_reshaped = blocks_batch[:, None, :]
        codewords_reshaped = codewords[None, :, :]

        # Compute euclidean distance between blocks and codewords
        distances = cp.linalg.norm(blocks_reshaped - codewords_reshaped, axis=2)

        # Assign each block to the closest codeword
        assignments[i:i+batch_size] = cp.argmin(distances, axis=1)

    return assignments



### Updates the codewords by replacing them with the average of the blocks that were
### assigned to them. If no blocks were assigned to a codeword, keeps the old one.
def update_codewords(blocks, assignments, num_codewords, old_codewords, block_dim):

    new_codewords = cp.zeros((num_codewords, block_dim**3))
    counts = cp.zeros(num_codewords)

    # Accumulate blocks into assigned codewords
    cp.add.at(new_codewords, assignments, blocks)
    cp.add.at(counts, assignments, 1)

    # Avg the codewords
    nonzero_counts = counts > 0
    new_codewords[nonzero_counts] /= counts[nonzero_counts][:, None]
    new_codewords[~nonzero_counts] = old_codewords[~nonzero_counts]

    return new_codewords



### Performs the Generalized Lloyd Algorithm (GLA) to generate a codebook based of size
### codebook_size for a group of blocks (blocks). Runs until algorithm reaches
### max_iter, or until converges: sum of changes to codewords is < tol.
def gla(max_iter, blocks, codebook_size, tol):

    # Compile blocks from all levels, boxes into one list, and translate to CuPy array
    all_blocks_list = [block for blocks_l in blocks for sub_blocks in blocks_l for block in sub_blocks]
    all_blocks = cp.empty((len(all_blocks_list), block_dim**3), dtype=cp.float32)
    all_blocks[:] = cp.array(all_blocks_list, dtype=cp.float32)

    codewords = cp.array(initialize_codewords_random(all_blocks, codebook_size))
    print("Initialized codewords. Beginning VQ.")
    prev_codewords = cp.array(codewords)

    # Perform GLA
    for iteration in tqdm(range(max_iter)):
        # Assign blocks to nearest codewords
        assignments = assign_to_codeword(all_blocks, codewords)
        # Update codewords according to assignments
        codewords = update_codewords(all_blocks, assignments, codebook_size, prev_codewords, block_dim)

        # Check for convergence
        delta = cp.sum(cp.linalg.norm(codewords - prev_codewords, axis=1))
        if delta < tol:
            print(f"Converged after {iteration + 1} iterations.")
            break

        # Update prev_codewords before next iteration
        prev_codewords = codewords

    # Re-format assignments for ease of query based on level, box_idx, etc.
    final_assignments = []
    all_blocks_idx = 0
    for l in range(len(blocks)):
        assignments_l = []
        blocks_l = blocks[l]
        for box_idx in range(len(blocks_l)):
            sub_assignments = []
            sub_blocks = blocks_l[box_idx]
            for block_idx in range(len(sub_blocks)):
                sub_assignments.append(assignments[all_blocks_idx])
                all_blocks_idx += 1
            assignments_l.append(sub_assignments)
        final_assignments.append(assignments_l)
    
    codebook = (final_assignments, codewords)

    return codebook



def find_block(xdim, ydim, zdim, block_dim, x, y, z):

    x_steps = int(xdim // block_dim)
    if xdim % block_dim != 0:
        x_steps += 1
    if xdim < block_dim:
        x_steps = 1
    y_steps = int(ydim / block_dim)
    if ydim % block_dim != 0:
        y_steps += 1
    if ydim < block_dim:
        y_steps = 1
    z_steps = int(zdim / block_dim)
    if zdim % block_dim != 0:
        z_steps += 1
    if zdim < block_dim:
        z_steps = 1

    z_block = (z // block_dim)
    z_rem = z % block_dim
    y_block = (y // block_dim)
    y_rem = y % block_dim
    x_block = (x // block_dim)
    x_rem = x % block_dim

    block_idx = (z_block * (y_steps * x_steps)) + (y_block * x_steps) + x_block
    rem = (x_rem, y_rem, z_rem)

    return block_idx, rem

    

### Reconstructs a box of a given dimension (dim) using a ...
def reconstruct_box(dim, block_dim, codewords, assignments):

    xdim, ydim, zdim = dim[0], dim[1], dim[2]
    
    reconstructed = np.zeros((xdim, ydim, zdim))
    
    for z in range(zdim):
        for y in range(ydim):
            for x in range(xdim):
                
                block_idx, rem = find_block(xdim, ydim, zdim, block_dim, x, y, z)
                codebook_idx = assignments[block_idx]
                codeword = codewords[int(codebook_idx)]
                
                codeword = codeword.reshape(block_dim, block_dim, block_dim)
                pix = codeword[rem[0], rem[1], rem[2]]
                reconstructed[x][y][z] = pix

    return reconstructed



### Calculates the average root mean squared error for a given list of boxes
### and their regenerations
def calc_avg_rmse(actuals, regens):

    rmses = []

    for l in range(len(actuals)): # iterate over levels
        boxes_l = actuals[l]
        regen_boxes_l = regens[l]
    
        for box_idx in range(len(boxes_l)): # iterate through boxes at current level

            actual = boxes_l[box_idx]
            pred = regen_boxes_l[box_idx]
        
            xdim = actual.shape[0]
            ydim = actual.shape[1]
            zdim = actual.shape[2]

            sum = 0
        
            for k in range(zdim):
                for j in range(ydim):
                    for i in range(xdim):
                        sq = (actual[i][j][k] - pred[i][j][k])**2
                        sum += sq

            rmse = (sum / (xdim * ydim * zdim))**0.5
            rmses.append(rmse)

    rmses = np.array(rmses)
    return rmses.mean()

#### Hyperparameters

In [35]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on", device)
# min_file = 74
# max_file = 75
timestep = 74
min_level = 0
max_level = 0
block_dim = 2
# level = 3
component = 6
junk = 0
max_iter = 100
tol = 1e-4
codebook_size = 4096
dir_prefix = "wholeVolumesNewFormat-" + str(component)

Running on cuda


#### Data preprocessing

In [39]:
boxes, locations, dimensions, num_boxes = process_data(dir_prefix, timestep, min_level, max_level, component)

blocks = create_samples(boxes, block_dim, junk)

Processing data...


100%|█████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.09it/s]


Creating samples...


100%|█████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.20it/s]


#### Generalized Llyod Algorithm to create codebook

In [40]:
codebook = gla(max_iter, blocks, codebook_size, tol)

# Check size of codebook
assignments_size = 0
for lev in range(len(codebook[0])):
    for box_idx in range(len(codebook[0][0])):
        for block_idx in range(len(codebook[0][0][0])):
            assignments_size += codebook[0][lev][box_idx][block_idx].nbytes
codewords_size = codebook[1].nbytes
print("Codebook size:", assignments_size + codewords_size)

Initialized codewords. Beginning VQ.


100%|█████████████████████████████████████████████████████████████████████████████████| 100/100 [05:49<00:00,  3.50s/it]


Codebook size: 1441792


#### Reconstruct timestep from codebook

In [41]:
final_assignments = codebook[0]`
codewords = codebook[1]

regen_boxes = []

for l in range(len(blocks)): # iterating over level

    dimensions_l = dimensions[l]
    assignments_l = final_assignments[l]
    regen_boxes_l = [None] * len(dimensions_l)

    for box_idx in tqdm(range(len(dimensions_l))):
        regen_boxes_l[box_idx] = reconstruct_box(dimensions_l[box_idx],
                                block_dim, codewords, assignments_l[box_idx])

    regen_boxes.append(regen_boxes_l)

rmse = calc_avg_rmse(boxes, regen_boxes)
print(rmse)

100%|█████████████████████████████████████████████████████████████████████████████████| 576/576 [01:11<00:00,  8.09it/s]


13.756911359983414


#### Write reconstructed data to binary file for visualization

In [ ]:
### Identical to write_to_bin, but for writing a decoded box to a binary
### file with the first six floats consisting of the location and dimension
### of the box.
def write_to_bin_with_loc_dim(volume, path, out_file, location, dimension):
    filename = path + out_file
    with open(filename, "wb") as file:
        # Write location
        for coord in range(len(location)):
            packed = struct.pack("<f", location[coord])
            file.write(packed)

        # Write dimension
        for coord in range(len(dimension)):
            packed = struct.pack("<f", dimension[coord])
            file.write(packed)

        # Write data
        for z in range(volume.shape[2]):
            for column in range(volume.shape[1]):
                for row in range(volume.shape[0]):
                    packed = struct.pack("<f", volume[row, column, z])
                    file.write(packed)



for l in range(len(regen_boxes)): # iterate through each level

    regen_boxes_l = regen_boxes[l]
    locations_l = locations[l]
    dimensions_l = dimensions[l]
        
    for i in tqdm(range(len(regen_boxes_l))): # iterate through the boxes at the current level
            
        # get the data for the current box
        box = regen_boxes_l[i]
        loc = locations_l[i]
        dim = dimensions_l[i]
            
        # convert to np array
        box = np.array(box, dtype="float32")

        # write current box to binary file, including time, location, and dimension info
        path = "decoded-" + str(l) + "/"
        filename = "decodedBox-" + str(timestep-74) + "-" + str(i) + ".raw"
        write_to_bin_with_loc_dim(box, path, filename, loc, dim)
